We use of SymPy in SageMath.

# 1. Series Expansions

If we use sympy in SageMath, then we may have to be explicit about the use of its functions.
For example, the ``sqrt()`` function in SageMath is different from ``the sqrt()`` function in sympy.
Consider the generating function for the Catalan numbers ``(1 - sqrt(4-x**2))/(2*x)``.
With sympy we can define a generator for the terms in a Taylor series,
but we must use the ``sqrt()`` function of sympy.  
But it starts sooner, already with the type of variables.

In [1]:
import sympy
x = sympy.var('x')
type(x)

<class 'sympy.core.symbol.Symbol'>

Observe the difference with the var() that we are normally using in SageMath.

In [2]:
y = var('y')
type(y)

<class 'sage.symbolic.expression.Expression'>

Even if we use the sqrt() on a variable that is a sympy symbol, 
we get a symbolic expression of Sage on return.

In [3]:
type(sqrt(x))

<class 'sage.symbolic.expression.Expression'>

In [4]:
type(sqrt(y))

<class 'sage.symbolic.expression.Expression'>

We can make a generator object on the ``sqrt(1-x)``.  That works.

In [5]:
tsqrtx = sympy.series(sqrt(1-x),x,0,n=None)
tsqrtx

<generator object Expr.series.<locals>.<genexpr> at 0x7f587afddaf0>

In [6]:
next(tsqrtx)

1

In [7]:
next(tsqrtx)

-x/2

In [8]:
[next(tsqrtx) for _ in range(8)]

[-x**2/8,
 -x**3/16,
 -5*x**4/128,
 -7*x**5/256,
 -21*x**6/1024,
 -33*x**7/2048,
 -429*x**8/32768,
 -715*x**9/65536]

But it will not work on ``sqrt(1-y)``.

In [9]:
tsqrty = sympy.series(sqrt(1-y),y,0,n=None)

In [10]:
type(tsqrty)

<class 'generator'>

In [11]:
next(tsqrty)

sqrt(1 - y)

Doing another ``next(tsqrty)`` triggers the ``StopIteration`` exception.

Although we can explitly use the sqrt() of sympy,
it does make no difference, because ``y`` is a SageMath variable.

So for the generating function for the Catalan numbers, 
we must use a sympy symbol x and not a Sage variable y.

In [12]:
g = (1 - sqrt(1-4*x))/(2*x)
cg = sympy.series(g,x,0,n=None)
[next(cg) for k in range(11)]

[1,
 x,
 2*x**2,
 5*x**3,
 14*x**4,
 42*x**5,
 132*x**6,
 429*x**7,
 1430*x**8,
 4862*x**9,
 16796*x**10]

In [13]:
catalan_number(10)

16796

# 2. Pattern Matching

Another feature that is a bit better in SymPy than in Sage is pattern matching,
which can be useful when combined with substitution.
To run this code we can also open a Terminal session, select Misc, and select python.

In [14]:
reset()
from sympy import *
x = Symbol('x')
y = Wild('y')
d = (10*x**3).match(y*x**3)
d

{y_: 10}

# 3. Solving Recurrence Relations

SymPy can solve recurrence relations.
Consider the Fibonacci numbers,
defined as $f(n) = f(n-1) + f(n-2)$.

In [15]:
reset()
from sympy import Function, rsolve
from sympy.abc import n
f = Function('f')
Fib = f(n+2) - f(n+1) - f(n)
Fib

-f(n) - f(n + 1) + f(n + 2)

To solve a recurrence relation, we call ``rsolve``.

In [16]:
s = rsolve(Fib, f(n))
s

C0*(1/2 - sqrt(5)/2)**n + C1*(1/2 + sqrt(5)/2)**n

Because we have a two terms relation,
for a unique solution, we need to initial conditions,
as an additional dictionary argument.

In [17]:
s2 = rsolve(Fib, f(n), {f(0): 0, f(1): 1})
s2

-sqrt(5)*(1/2 - sqrt(5)/2)**n/5 + sqrt(5)*(1/2 + sqrt(5)/2)**n/5

Let us now compute the first 10 Fibonacci numbers.  For this, we return to Sage and convert the SymPy object into a symbolic expression with ``SR()``.

In [18]:
type(s2)

<class 'sympy.core.add.Add'>

In [19]:
fs2 = SR(s2)
type(fs2)

<class 'sage.symbolic.expression.Expression'>

Now we can evaluate and expand the ``sqrt(5)`` expressions,
which are then automatically simplified.

In [20]:
[fs2(n=k).expand() for k in range(10)]

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

# 4. Finding the Nearest Algebraic Number

With ``PSLQ`` one can compute the nearest algebraic number to a given floating-point approximation.  This is available in ``mpmath``.

In [21]:
s2 = RR(sqrt(2))
s2

1.41421356237310

In [22]:
from mpmath import pslq

In [23]:
pwrs = [s2**k for k in range(3)]
pwrs

[1.00000000000000, 1.41421356237310, 2.00000000000000]

In [24]:
cff = pslq(pwrs, verbose=True)
cff

PSLQ using prec 113 and tol 1.81899e-12
0/100:  Error:     0.06   Norm: 0
FOUND relation at iter 1/100, error: 2.0e-16


[2, 0, -1]

In ``cff`` are the coefficients of the polynomial ``2 - x^2``, which is the smallest polynomial for which ``s2`` is an approximate root.